# 📊 Chapter 4: Conditional Probability
*Tier 1 — Foundations | All Tracks*

---
> **🎬 Hook:** A medical test for a rare disease is 99% accurate.
> You test positive. Are you 99% likely to have the disease?
> The answer is **NO** — and it might be less than 10%.
> This is the most important (and most misunderstood) concept in probability.

**🎯 Learning Objectives**
- Calculate conditional probability P(A|B) correctly
- Understand what "given that" means mathematically
- Spot independence vs dependence
- Avoid the Prosecutor's Fallacy and Base Rate Neglect

## 📖 Section 1 — Concept Review

### What is Conditional Probability?

P(A|B) = "Probability of A, **given that** B has already happened"

When we learn that B occurred, it **shrinks our sample space** from Ω to just B.
Then we ask: within B, what fraction of outcomes are also in A?

$$P(A|B) = \frac{P(A \cap B)}{P(B)} \quad \text{(assuming P(B) > 0)}$$

### Visual Intuition: Restricting the Sample Space

Before knowing B: We look at ALL of Ω
After knowing B: We zoom in — B is now our entire world

```
     Ω                    B (our new world)
  ┌──────────┐         ┌────────┐
  │    A  ╔══╪══╗      │   A∩B  │
  │   ╔══╪═╪══╗│      │        │
  │   ║  │ ║  ║│  →   └────────┘
  │   ╚══╪═╪══╝│      P(A|B) =
  │      ╚══╪══╝      |A∩B| / |B|
  └──────────┘
```

### Independence
A and B are **independent** if knowing B tells you NOTHING about A:
$$P(A|B) = P(A) \quad \Leftrightarrow \quad P(A \cap B) = P(A) \cdot P(B)$$

### ⚠️ Common Traps
1. **Base Rate Neglect:** Ignoring how rare the disease is before the test
2. **Prosecutor's Fallacy:** Confusing P(evidence|innocent) with P(innocent|evidence)
3. **Confusion of the inverse:** P(A|B) ≠ P(B|A)

## 🧮 Section 2 — Math Walkthrough

### The Medical Test Problem

- Disease prevalence (base rate): P(Disease) = 0.001 (1 in 1,000)
- Test sensitivity: P(Positive | Disease) = 0.99
- Test specificity: P(Negative | No Disease) = 0.99 → P(Positive | No Disease) = 0.01

**What we want:** P(Disease | Positive test) = ?

Using the definition:
$$P(D|+) = \frac{P(+ \cap D)}{P(+)}$$

**Step 1:** P(+ and Disease) = P(+|D) × P(D) = 0.99 × 0.001 = **0.00099**

**Step 2:** P(+) = P(+|D)×P(D) + P(+|¬D)×P(¬D)
= 0.99×0.001 + 0.01×0.999 = 0.00099 + 0.00999 = **0.01098**

**Step 3:**
$$P(D|+) = \frac{0.00099}{0.01098} \approx \mathbf{0.090 = 9\%}$$

**Despite a 99% accurate test, a positive result means only a 9% chance of disease!**
This is called the **False Positive Paradox** — and it's caused by base rate neglect.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style="whitegrid")
np.random.seed(42)

# The Medical Test Problem — visualized
def medical_test_analysis(prevalence, sensitivity, specificity):
    fp_rate = 1 - specificity

    # Bayes' theorem
    p_pos_and_disease   = sensitivity * prevalence
    p_pos_and_no_disease = fp_rate * (1 - prevalence)
    p_positive           = p_pos_and_disease + p_pos_and_no_disease
    p_disease_given_pos  = p_pos_and_disease / p_positive

    return {
        'P(D)': prevalence,
        'P(+|D)': sensitivity,
        'P(+|¬D)': fp_rate,
        'P(+)': p_positive,
        'P(D|+)': p_disease_given_pos
    }

# Scenario: common vs rare disease
scenarios = [
    {"name": "Rare disease (1 in 1000)", "prev": 0.001, "sens": 0.99, "spec": 0.99},
    {"name": "Common disease (1 in 10)", "prev": 0.10,  "sens": 0.99, "spec": 0.99},
    {"name": "Very common (1 in 3)",     "prev": 0.33,  "sens": 0.99, "spec": 0.99},
]

print("🏥 Medical Test Analysis: P(Disease | Positive Test)")
print("=" * 65)
for s in scenarios:
    result = medical_test_analysis(s['prev'], s['sens'], s['spec'])
    print(f"\n{s['name']}")
    print(f"  Base rate:          {s['prev']:.3f}")
    print(f"  P(Disease | +test): {result['P(D|+)']:.3f} ({result['P(D|+)']:.1%})")

print("\n💡 KEY INSIGHT: Same 99% accurate test — but VERY different results based on base rate!")

In [ ]:
# Visualize how base rate affects P(Disease | Positive)
prevalences = np.linspace(0.001, 0.5, 200)
p_d_given_pos = []
for prev in prevalences:
    r = medical_test_analysis(prev, 0.99, 0.99)
    p_d_given_pos.append(r['P(D|+)'])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(prevalences, p_d_given_pos, lw=3, color='#e74c3c', label='P(Disease | Positive test)')
ax.axhline(0.5, color='gray', linestyle=':', lw=1.5, label='50% threshold')
ax.fill_between(prevalences, p_d_given_pos, alpha=0.2, color='#e74c3c')

# Mark our key scenarios
for prev, label in [(0.001, 'Rare: 0.1%'), (0.10, 'Common: 10%'), (0.33, 'Very common: 33%')]:
    r = medical_test_analysis(prev, 0.99, 0.99)
    ax.scatter([prev], [r['P(D|+)']], s=100, zorder=5, color='black')
    ax.annotate(f"{label}\n→ P(D|+)={r['P(D|+)']:.1%}",
                xy=(prev, r['P(D|+)']), xytext=(prev+0.03, r['P(D|+)']-0.08),
                fontsize=9, arrowprops=dict(arrowstyle='->', color='black'))

ax.set_xlabel('Disease Prevalence (Base Rate)', fontsize=12)
ax.set_ylabel('P(Disease | Positive Test)', fontsize=12)
ax.set_title('🏥 The Base Rate Effect: How Rare the Disease is Changes Everything', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('ch04_base_rate.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔬 Section 3 — Simulation: The Monty Hall Problem

The most famous conditional probability puzzle:
You're on a game show. 3 doors: one has a car, two have goats.
You pick door 1. The host (who knows where the car is) opens door 3 showing a goat.
Should you switch to door 2?

**Most people say: doesn't matter, 50/50.**
**Math says: SWITCH — P(win|switch) = 2/3!**

In [ ]:
def monty_hall_simulation(n=100_000, switch=True):
    wins = 0
    for _ in range(n):
        car_door = np.random.randint(0, 3)
        chosen   = np.random.randint(0, 3)

        # Host opens a door that has a goat and isn't chosen
        remaining = [d for d in range(3) if d != chosen and d != car_door]
        host_opens = np.random.choice(remaining)

        if switch:
            # Switch to the remaining door
            chosen = [d for d in range(3) if d != chosen and d != host_opens][0]

        wins += (chosen == car_door)
    return wins / n

p_win_stay   = monty_hall_simulation(100_000, switch=False)
p_win_switch = monty_hall_simulation(100_000, switch=True)

print("🚗 Monty Hall Problem Simulation (100,000 games)")
print(f"  P(win | STAY)   = {p_win_stay:.4f}  (theory: 1/3 = {1/3:.4f})")
print(f"  P(win | SWITCH) = {p_win_switch:.4f}  (theory: 2/3 = {2/3:.4f})")
print()
print("💡 ALWAYS SWITCH! Your initial guess is wrong 2/3 of the time.")
print("   The host's action gives you NEW INFORMATION — use it!")

# Visualize
fig, ax = plt.subplots(figsize=(7, 4))
strategies = ['Stay', 'Switch']
probs = [p_win_stay, p_win_switch]
colors = ['#e74c3c', '#27ae60']
bars = ax.bar(strategies, probs, color=colors, width=0.4, edgecolor='white', linewidth=2)
ax.axhline(1/3, color='gray', linestyle='--', alpha=0.5, label='1/3')
ax.axhline(2/3, color='gray', linestyle=':', alpha=0.5, label='2/3')
for bar, p in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{p:.1%}', ha='center', fontsize=14, fontweight='bold')
ax.set_ylim(0, 0.8)
ax.set_ylabel('P(Win)')
ax.set_title('🚗 Monty Hall: Always Switch!', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ch04_monty_hall.png', dpi=150, bbox_inches='tight')
plt.show()

## 📊 Section 4 — Visualization: Conditional Probability Table

In [ ]:
# Contingency table visualization
np.random.seed(42)
n = 10_000

# Gender and sport preference (hypothetical data)
is_female = np.random.random(n) < 0.5
likes_soccer = np.where(is_female,
                        np.random.random(n) < 0.45,   # P(soccer|female) = 0.45
                        np.random.random(n) < 0.65)   # P(soccer|male) = 0.65

# Build contingency table
df = pd.DataFrame({'Gender': np.where(is_female, 'Female', 'Male'),
                   'Sport': np.where(likes_soccer, 'Soccer', 'Other')})
ct = pd.crosstab(df['Gender'], df['Sport'], margins=True, normalize=False)
ct_pct = pd.crosstab(df['Gender'], df['Sport'], normalize='index')

print("📊 Contingency Table (Counts)")
print(ct)
print()
print("📊 Conditional Probabilities P(Sport | Gender)")
print(ct_pct.round(3))
print()
print(f"P(Soccer | Female) = {ct_pct.loc['Female', 'Soccer']:.3f}")
print(f"P(Soccer | Male)   = {ct_pct.loc['Male', 'Soccer']:.3f}")
print(f"P(Soccer)          = {likes_soccer.mean():.3f}  ← marginal probability")
print()
print("Since P(Soccer|Female) ≠ P(Soccer|Male), Gender and Sport preference are NOT independent!")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ct_pct.plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c'], rot=0)
axes[0].set_title('Conditional: P(Sport | Gender)', fontweight='bold')
axes[0].set_ylabel('Probability')
axes[0].legend(title='Sport')

# Joint probabilities
ct_joint = pd.crosstab(df['Gender'], df['Sport'], normalize=True)
ct_joint.plot(kind='bar', ax=axes[1], color=['#3498db', '#e74c3c'], rot=0)
axes[1].set_title('Joint: P(Gender, Sport)', fontweight='bold')
axes[1].set_ylabel('Probability')

plt.tight_layout()
plt.savefig('ch04_contingency.png', dpi=150, bbox_inches='tight')
plt.show()

## 📂 Section 5 — Real Dataset Exercise: Spam Filter

Email spam filters use conditional probability.
P(spam | contains "FREE") vs P(spam | contains "meeting")

In [ ]:
# Simulate spam filter scenario
np.random.seed(42)
n_emails = 5000
p_spam = 0.3

is_spam = np.random.random(n_emails) < p_spam

# Word "FREE": appears in 60% of spam, 5% of legitimate emails
has_free = np.where(is_spam, np.random.random(n_emails) < 0.60,
                    np.random.random(n_emails) < 0.05)

# Word "meeting": appears in 5% of spam, 40% of legitimate emails
has_meeting = np.where(is_spam, np.random.random(n_emails) < 0.05,
                        np.random.random(n_emails) < 0.40)

# Calculate conditional probabilities
p_spam_given_free    = (is_spam & has_free).sum() / has_free.sum()
p_spam_given_meeting = (is_spam & has_meeting).sum() / has_meeting.sum()
p_spam_given_neither = (is_spam & ~has_free & ~has_meeting).sum() / (~has_free & ~has_meeting).sum()

print("📧 Email Spam Filter: Conditional Probabilities")
print(f"  Prior: P(spam) = {is_spam.mean():.3f}")
print()
print(f"  P(spam | contains 'FREE')    = {p_spam_given_free:.3f}  ← very suspicious!")
print(f"  P(spam | contains 'meeting') = {p_spam_given_meeting:.3f}  ← likely legitimate")
print(f"  P(spam | neither word)       = {p_spam_given_neither:.3f}")
print()
print("💡 This is Naive Bayes classification — used in every spam filter!")

words = ["Contains 'FREE'", "Contains 'meeting'", "Neither word"]
spam_probs = [p_spam_given_free, p_spam_given_meeting, p_spam_given_neither]
colors = ['#e74c3c' if p > 0.5 else '#27ae60' for p in spam_probs]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(words, spam_probs, color=colors, alpha=0.8)
ax.axvline(0.5, color='black', linestyle='--', lw=2, label='Decision threshold (0.5)')
ax.axvline(p_spam, color='gray', linestyle=':', lw=2, label=f'Prior P(spam) = {p_spam}')
ax.set_xlabel('P(Spam | Word)')
ax.set_title('📧 Spam Filter: Conditional Probability Updates Belief', fontweight='bold')
ax.legend()
for bar, p in zip(bars, spam_probs):
    ax.text(p + 0.01, bar.get_y() + bar.get_height()/2, f'{p:.3f}', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('ch04_spam_filter.png', dpi=150, bbox_inches='tight')
plt.show()

## ✏️ Section 6 — Practice Problems

**Problem 1 — Basic:**
P(A) = 0.4, P(B) = 0.3, P(A∩B) = 0.12.
Find P(A|B) and P(B|A). Are A and B independent?

**Problem 2 — Medical:**
A disease affects 2% of a population. A test has 95% sensitivity and 90% specificity.
What is P(disease | positive test)?

**Problem 3 — Challenge (Prosecutor's Fallacy):**
A crime was committed. DNA evidence matches the suspect with P = 1/1,000,000.
Prosecutor argues: "P(innocent | match) = 0.000001"
Why is this WRONG? What do we actually need to know?

---
<details>
<summary>💡 Solutions</summary>

**P1:** P(A|B) = P(A∩B)/P(B) = 0.12/0.30 = **0.40**
P(B|A) = P(A∩B)/P(A) = 0.12/0.40 = **0.30**
Since P(A|B) = P(A) = 0.40, they ARE independent.

**P2:** P(D|+) = (0.95×0.02) / (0.95×0.02 + 0.10×0.98) = 0.019/(0.019+0.098) = **16.2%**

**P3:** The prosecutor confused P(match|innocent) with P(innocent|match).
We need P(innocent) = 1 - P(guilty before DNA evidence). If only 1 person in a city of 1M could have done it, P(match|innocent) = 1/1M seems damning. But it needs prior probability applied correctly.
</details>

In [ ]:
# Verify Problem 2
prevalence = 0.02
sensitivity = 0.95
specificity = 0.90
fp_rate = 1 - specificity

p_pos_given_disease    = sensitivity
p_pos_given_no_disease = fp_rate

p_pos = p_pos_given_disease * prevalence + p_pos_given_no_disease * (1 - prevalence)
p_disease_given_pos = (p_pos_given_disease * prevalence) / p_pos

print(f"Problem 2 Solution:")
print(f"  P(D)       = {prevalence}")
print(f"  P(+|D)     = {sensitivity}")
print(f"  P(+|¬D)    = {fp_rate}")
print(f"  P(+)       = {p_pos:.4f}")
print(f"  P(D|+)     = {p_disease_given_pos:.4f} = {p_disease_given_pos:.1%}")
print()
print(f"Even with a 95% sensitive test, only {p_disease_given_pos:.1%} of positives actually have the disease!")
print("This is because the disease is RARE — base rate matters enormously.")

## 🎯 Episode Recap

**3 Key Takeaways:**
1. **P(A|B) means we restrict our sample space to B** — we're no longer considering all outcomes.
2. **Base Rate Neglect is everywhere** — always consider how rare/common an event is before interpreting a test.
3. **P(A|B) ≠ P(B|A)** — the direction of conditioning completely changes the answer.

**🔗 Next Episode:** [Chapter 5 — Bayes' Theorem: Prior, Likelihood, and Posterior]

**💬 Viewer Challenge:** Solve the Birthday Problem: What's the minimum number of people in a room for P(at least two share a birthday) > 50%? (Hint: use the complement rule!)